# 2. tokenizer2:

### Learning Objectives
By the end of this lesson, you should be able to:

* Build a production-ready BPE Tokenizer that correctly handles Unicode, whitespace normalization, and special tokens.
* Implement byte-level fallback so the Tokenizer can encode any input, including emojis, CJK text, and code, without generating unknown tokens.
* Use a pre-tokenization regex to split text at appropriate word, number, punctuation, and whitespace boundaries before executing BPE merges.
* Train a custom Tokenizer on a corpus and compare its compression ratio on multilingual text with `tiktoken`.
* Understand the role of Chat Templates in converting structured messages into Token IDs.
* Explain the differences between an educational Python implementation and a production-ready Tokenizer in terms of speed, accuracy, and reproducibility.
---
### What is the Problem?

The BPE Tokenizer from Lesson 01 worked on English text. Now test that same Tokenizer with Japanese text, emojis, or Python code containing a mix of tabs and spaces; part of the process will likely break or produce an unsuitable output.

The problem is not with the BPE algorithm itself; the problem is that the implementation is not yet complete. A production-ready Tokenizer must:

- Handle input at the byte level, independent of language;
- Normalize Unicode according to a defined policy before splitting text;
- Have special tokens that are never split or merged with other tokens;
- Combine pre-tokenization with subword splitting;
- Provide reliable, and ideally reversible, encode and decode capabilities;
- Be fast enough not to become a bottleneck in the training pipeline;
- Save vocabulary, merge rules, normalization, and special token configurations in a versioned format so results are reproducible.

The GPT-2 vocabulary contains 50,257 tokens, and Llama 3 uses a vocabulary of 128,256 tokens. For GPT-4 family models, tokenizers typically employ vocabularies on the scale of approximately 100,000 tokens; however, the exact number depends on the model and the encoding used.

These numbers do not belong to small toy examples. The merge tables for such vocabularies are trained on massive volumes of data. Beyond BPE itself, components such as normalization, pre-tokenization, special token handling, and chat template formatting separate a tokenizer limited to a "hello world" phrase from one suitable for extensive internet-scale data.

In this lesson, you will build and understand these very components and the logic behind them.

---

### Core Concept: The Full Pipeline

A production-ready Tokenizer is not just a single algorithm; it is a pipeline composed of several stages, each solving a different problem.

    A[Raw Text] --> B[Normalize] --> C[Pre-tokenize]--> D[BPE Merge]--> E[Special Tokens]--> F[Token IDs]





Packages

In [22]:
import re
import unicodedata
from collections import Counter
from typing import Dict, List, Tuple, Union

In [23]:
import regex

PATTERN = regex.compile(
    r"'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+",
    flags=regex.IGNORECASE,
)

def pre_tokenize(text: str) -> list[str]:
    return PATTERN.findall(text)


print(pre_tokenize("I don't code in Python 3.11!"))

['I', ' don', "'t", ' code', ' in', ' Python', ' 3', '.', '11', '!']


This code is responsible for pre-tokenization based on the standard GPT-2 pattern.

This algorithm splits the text into smaller chunks prior to BPE to prevent consecutive words or punctuation marks from merging:


In [24]:
# create pattern
# if have `regex`, use that but have not use `re`
try:
    import regex
    GPT2_PATTERN = regex.compile(
        r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    )
except ImportError:
    GPT2_PATTERN = re.compile(
        r"""'(?:[sdmt]|ll|ve|re)| ?[a-zA-Z]+| ?[0-9]+| ?[^\s\w]+|\s+(?!\S)|\s+"""
    )



## part 1: `def pre_tokenize`

To implement the `pre_tokenize` method, we must use `findall` with `GPT2_PATTERN` to extract all segments matching the regex rules as a list of strings.


In [25]:
# part 1----------------------------------------------
def pre_tokenize(text: str) -> List[str]:
    """
    Split input text into initial word/symbol chunks using the GPT-2 regex pattern.

    Args:
        text (str): Raw input text string to pre-tokenize.

    Returns:
        List[str]: A list of string chunks matched by the pre-tokenization regex.
    """
    # TODO: Apply GPT2_PATTERN regex iterator over text to extract all chunk string matches
    return GPT2_PATTERN.findall(text)


In [26]:
# manual test
print("/1/".center(40, '-'))
sample_1 = "I am Mohsen Mohebbi. I participated in the Daneshkar Artificial Intelligence course."
print(f"split form:---------------------\n{(sample_1.split())}")
print(f"use function pre_tokenize:------\n{pre_tokenize(sample_1)}")


------------------/1/-------------------
split form:---------------------
['I', 'am', 'Mohsen', 'Mohebbi.', 'I', 'participated', 'in', 'the', 'Daneshkar', 'Artificial', 'Intelligence', 'course.']
use function pre_tokenize:------
['I', ' am', ' Mohsen', ' Mohebbi', '.', ' I', ' participated', ' in', ' the', ' Daneshkar', ' Artificial', ' Intelligence', ' course', '.']


## part 2: `def apply_merge`




In [27]:
# Part 2----------------------------------------------
def apply_merge(byte_seq: List[int], pair: Tuple[int, int], new_id: int) -> List[int]:
    """
    Replace consecutive occurrences of a specific pair of token IDs in a sequence with a new token ID.

    Args:
        byte_seq (List[int]): Current sequence of token IDs.
        pair (Tuple[int, int]): A tuple (first_id, second_id) representing the pair to merge.
        new_id (int): The new token ID assigned to the merged pair.

    Returns:
        List[int]: A new list of token IDs with target pairs merged.
    """
    # TODO: Iterate through byte_seq, find adjacent matching pairs, and replace them with new_id
    # if len byte<2 , NOT merge
    if len(byte_seq) < 2:
        return list(byte_seq)

    # make merge list
    merged: list[int] = []
    i = 0
    first, second = pair

    while i < len(byte_seq):
        # len of text is end? 
        if i < len(byte_seq) - 1 and byte_seq[i] == first and byte_seq[i + 1] == second:
            merged.append(new_id)
            i += 2  # go to next pair
        else:
            merged.append(byte_seq[i])
            i += 1

    return merged


In [28]:
# manual test
print("/1/".center(40, '-'))
sample_1 = "I am Mohsen Mohebbi. I participated in the Daneshkar Artificial Intelligence course."
# use pre_tokenize
test_chunks = pre_tokenize(sample_1)
print(f"main text:-------------------\n{sample_1}")
print(f"first use pre_tokenize:------\n{test_chunks}")

print("/2/".center(40, '-'))
# we need encode
test_bytes = list(sample_1.encode("utf-8"))

if len(test_bytes) < 2:
    print("len byte < 2")


test_pair, test_new_id = (ord("M"), ord("o")), 999

merged: list[int] = []
i = 0
first, second = test_pair

while i < len(test_bytes):
    # len of text is end? 
    if i < len(test_bytes) - 1 and test_bytes[i] == first and test_bytes[i + 1] == second:
        merged.append(test_new_id)
        i += 2  # go to next pair
    else: # if we have 1 token and have not pair token
        merged.append(test_bytes[i])
        i += 1

print(f"merge is : \n{merged}")

print("/test_fucntion2/".center(40, '-'))
print(f"pair : {test_pair} and new_id : {test_new_id}")

test_apply_merge = apply_merge(test_bytes, test_pair, test_new_id)
print(f"def apply_merge --------------\n{test_apply_merge}")

------------------/1/-------------------
main text:-------------------
I am Mohsen Mohebbi. I participated in the Daneshkar Artificial Intelligence course.
first use pre_tokenize:------
['I', ' am', ' Mohsen', ' Mohebbi', '.', ' I', ' participated', ' in', ' the', ' Daneshkar', ' Artificial', ' Intelligence', ' course', '.']
------------------/2/-------------------
merge is : 
[73, 32, 97, 109, 32, 999, 104, 115, 101, 110, 32, 999, 104, 101, 98, 98, 105, 46, 32, 73, 32, 112, 97, 114, 116, 105, 99, 105, 112, 97, 116, 101, 100, 32, 105, 110, 32, 116, 104, 101, 32, 68, 97, 110, 101, 115, 104, 107, 97, 114, 32, 65, 114, 116, 105, 102, 105, 99, 105, 97, 108, 32, 73, 110, 116, 101, 108, 108, 105, 103, 101, 110, 99, 101, 32, 99, 111, 117, 114, 115, 101, 46]
------------/test_fucntion2/------------
pair : (77, 111) and new_id : 999
def apply_merge --------------
[73, 32, 97, 109, 32, 999, 104, 115, 101, 110, 32, 999, 104, 101, 98, 98, 105, 46, 32, 73, 32, 112, 97, 114, 116, 105, 99, 105, 112, 

## part 3: `class SpecialTokenHandler`

Manages registration and regex-based splitting of special tokens during tokenization

1. `def add_token`
   - Saving the string-to-ID mapping
   - Constructing a regex that ORs all registered tokens together, allowing us to split the text based on them.
   - Every time a new token is added, `self.pattern` is reconstructed. Using `re.escape` is crucial because tokens like `<|endoftext|>` contain regex reserved characters. This pattern will be used in the next method (`split_with_specials`) to locate the exact positions of special tokens in the raw text.

2. `def split_with_specials`



In [29]:
# Part 3---------------------------------------
class SpecialTokenHandler:
    """
    Manages registration and regex-based splitting of special tokens during tokenization.
    """

    def __init__(self) -> None:
        """Initialize empty special tokens mapping and pattern compiler."""
        self.special_tokens: Dict[str, int] = {}
        self.pattern: Union[re.Pattern, None] = None

    def add_token(self, token_str: str, token_id: int) -> None:
        """
        Register a special token and update the combined regular expression pattern.

        Args:
            token_str (str): The string representation of the special token (e.g., '<|end|>').
            token_id (int): The integer vocabulary ID assigned to the special token.

        Returns:
            None
        """
        # TODO: Store the token ID mapping and update the compiled regex pattern using re.escape
        # Save token and ID
        self.special_tokens[token_str] = token_id
        
        # Constructing the regex pattern for all registered special token `]`
        # Escaping special characters "<|end|>" ➜ "<\\|end\\|>"
        # `|` IS NOT `or` in special characters
        patterns = [re.escape(t) for t in self.special_tokens.keys()]
        
        # add `or` between special and next search that
        combined_pattern = "|".join(patterns)
        
        # fast to run
        # self.pattern is An object 
        self.pattern = re.compile(combined_pattern)

    def split_with_specials(self, text: str) -> List[Tuple[str, bool]]:
        """
        Segment text into a list of tuples containing text chunks and boolean flags indicating special tokens.

        Args:
            text (str): Input text that may contain special tokens.

        Returns:
            List[Tuple[str, bool]]: A list of tuples where each tuple is (substring, is_special_flag).
        """
        # TODO: Search text using pattern, split into standard text vs special token parts, and tag each part
        if not text:
            return []

        # if have NOT special characters return text
        if not self.pattern or not self.special_tokens:
            return [(text, False)]

        result: list[tuple[str, bool]] = []
        last_end = 0

        # find special characters and start and end
        for match in self.pattern.finditer(text):
            start, end = match.span()

            # between special characters have normal text
            if start > last_end:
                result.append((text[last_end:start], False))

            # add token
            result.append((match.group(), True))
            last_end = end

        # if next end special characters have text, add
        if last_end < len(text):
            result.append((text[last_end:], False))

        return result


In [30]:
# manual test
print("/1/".center(40, '-'))
test_Special = SpecialTokenHandler()
# add special characters
test_Special.add_token("<|endoftext|>", 50256) # special characters and token
test_Special.add_token("<|pad|>", 50257) # special characters and token

text_1 = "Hello world<|endoftext|>how are you?<|pad|>"
test_result = test_Special.split_with_specials(text_1)

print("test_result\n", test_result)


------------------/1/-------------------
test_result
 [('Hello world', False), ('<|endoftext|>', True), ('how are you?', False), ('<|pad|>', True)]


## part 4: `class ProductionTokenizer`

Byte-Pair Encoding (BPE) tokenizer supporting training, normalization, special tokens, encoding, and decoding

> 1. `__init__`
   - Initialize vocabulary, merges, special token handler, and next available token ID
> 2. `normalize`
   - use: form_norm = "NFKC"
> 3. `train`
   - Updating:
      - `self.merges`: The lookup table for all learned merges.
      - `self.vocab`: The lookup table mapping all tokens to their corresponding bytes
      - `self.next_id`: The next available token ID
> 4. `add_special_token`
   1. Allocate a new unique numerical ID (`token_id`) from `self.next_id` and increment the counter by one.
   2. Pass the special token string (e.g., `<|endoftext|>`) along with its ID to the `special_handler` object to register it in its regex pattern.
   3. Store the byte representation of the string (`token_str.encode("utf-8")`) in `self.vocab` keyed by this assigned ID.
   4. Return the allocated `token_id`.
> 5. `encode`
   - use normalizer from `def normalize` by `"NFKC"`
   - split from `class SpecialTokenHandler` -> `split_with_specials`
   - use fram `def pre_tokenize`
   - use from `def apply_merge`
> 6. `decode`
> 7. `vocab_size`
> 8. `get_token_bytes`



In [31]:
# part 4-------------------------------------------
class ProductionTokenizer:
    """
    Byte-Pair Encoding (BPE) tokenizer supporting training, normalization, special tokens, encoding, and decoding.
    """

    def __init__(self) -> None:
        """Initialize vocabulary, merges, special token handler, and next available token ID."""
        self.merges: Dict[Tuple[int, int], int] = {}
        self.vocab: Dict[int, bytes] = {i: bytes([i]) for i in range(256)} # [0, 255]
        # use class SpecialTokenHandler : Composition 
        self.special_handler: SpecialTokenHandler = SpecialTokenHandler()
        self.next_id: int = 256 # 255 +1 ...

    def normalize(self, text: str) -> str:
        """
        Normalize input text using Unicode NFKC normalization.

        Args:
            text (str): Raw input string.

        Returns:
            str: Unicode normalized string.
        """
        # TODO: Normalize input text using unicodedata NFKC standard
        # In the guide file:
        # Apply Unicode normalization such as NFKC, and if necessary, perform lowercasing or accent removal
        # but in todo say" `NFKC standard`
        # from package unicodedata use normalize
        # in doc `unicodedata.normalize` have `forms = ["NFC", "NFD", "NFKC", "NFKD"]`
        # https://docs.python.org/3/library/unicodedata.html
        form_norm = "NFKC"
        return unicodedata.normalize(form_norm, text)

    def train(self, text: str, num_merges: int) -> None:
        """
        Train the BPE tokenizer by identifying frequent adjacent pairs and iteratively merging them.

        Args:
            text (str): Corpus text used for training the tokenizer.
            num_merges (int): Number of BPE merge operations to execute.

        Returns:
            None
        """
        # TODO: Normalize and pre-tokenize corpus text into byte sequences
        # TODO: Iteratively count adjacent pair frequencies across all chunk byte sequences
        # TODO: Find the most frequent pair, create a new vocabulary entry, and record the merge rule
        # TODO: Replace the best pair in all chunk sequences using apply_merge
        # TODO: Normalize and pre-tokenize corpus text into byte sequences

        # use normalizer
        norm_text = self.normalize(text)

        # basic chunk
        # change to byte UTF-8 `[0, 255]`
        chunks = pre_tokenize(norm_text)
        chunk_ids: list[list[int]] = [list(chunk.encode("utf-8")) for chunk in chunks]

        # TODO: Iteratively count adjacent pair frequencies across all chunk byte sequences
        for _ in range(num_merges):
            # Pair Frequency Counting
            pair_counts: dict[tuple[int, int], int] = {}
            # make merge
            for ids in chunk_ids:
                for i in range(len(ids) - 1):
                    pair = (ids[i], ids[i + 1])
                    pair_counts[pair] = pair_counts.get(pair, 0) + 1

            #if len chunk < 2 stop
            if not pair_counts:
                break

            # TODO: Find the most frequent pair, create a new vocabulary entry, and record the merge rule
            best_pair = max(pair_counts, key=pair_counts.get)
            new_id = self.next_id
            self.next_id += 1

            # add to roll merge
            self.merges[best_pair] = new_id
            self.vocab[new_id] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]

            # TODO: Replace the best pair in all chunk sequences using apply_merge
            # edited `chunk_ids` to Reuse up to `num_merges`
            chunk_ids = [apply_merge(ids, best_pair, new_id) for ids in chunk_ids]

    def add_special_token(self, token_str: str) -> int:
        """
        Register a new special token into the tokenizer vocabulary and special token handler.

        Args:
            token_str (str): Special token string (e.g., '<|begin|>').

        Returns:
            int: Assigned vocabulary integer ID for the special token.
        """
        # TODO: Allocate new_id, register special token with special_handler and add byte representation to vocab
        token_id = self.next_id
        # first count token is 255 -> next new token is 255 + 1
        # '<|begin|>' -> 255 + 1
        self.next_id += 1

        # Registering the token in the special token handler engine for text segmentation
        # from class `special_handler`, add token
        # update Regex pattern
        self.special_handler.add_token(token_str, token_id)

        # add byte to dict for decoding
        self.vocab[token_id] = token_str.encode("utf-8")

        # This line returns the unique numerical ID assigned to the special token
        # Use for token padding and other part
        # If this ID is not returned, we cannt know ID
        return token_id

    def encode(self, text: str) -> List[int]:
        """
        Encode raw text into a sequence of vocabulary token IDs using trained merges and special tokens.

        Args:
            text (str): Input text string to be tokenized.

        Returns:
            List[int]: List of encoded vocabulary token IDs.
        """
        # TODO: Normalize input text and split into standard text and special token segments
        # TODO: For standard text segments, apply pre-tokenization and convert chunks into byte sequences
        # TODO: Apply learned BPE merges sequentially to byte sequences and collect all output token IDs

        #--------------------------------------------------------
        # TODO: Normalize input text and split into standard text and special token segments

        # use normalizer from `def normalize` by `"NFKC"`
        normalized_text = self.normalize(text)

        # split from `class SpecialTokenHandler` -> split_with_specials
        segments = self.special_handler.split_with_specials(normalized_text)

        tokens: List[int] = []

        for segment, is_special in segments: # is_special is true or false
            if is_special:
                # For special tokens, the ID is directly retrieved from the handler's dictionary
                tokens.append(self.special_handler.special_tokens[segment])
            else:
                # TODO: For standard text segments, apply pre-tokenization and convert chunks into byte sequences
                # fram `def pre_tokenize`
                chunks = pre_tokenize(segment)
                for chunk in chunks:
                    chunk_bytes = list(chunk.encode("utf-8"))

                    # TODO: Apply learned BPE merges sequentially to byte sequences and collect all output token IDs
                    for pair, new_id in self.merges.items():
                        # from `def apply_merge`
                        chunk_bytes = apply_merge(chunk_bytes, pair, new_id)

                    tokens.extend(chunk_bytes)

        return tokens

    def decode(self, ids: List[int]) -> str:
        """
        Decode a list of token IDs back into a UTF-8 string.

        Args:
            ids (List[int]): List of integer token IDs.

        Returns:
            str: Decoded UTF-8 text string.
        """
        # TODO: Map token IDs back to byte representations using vocabulary and decode as UTF-8
        byte_chunks = [self.vocab.get(i, b"") for i in ids]
        raw_bytes = b"".join(byte_chunks)
        return raw_bytes.decode("utf-8", errors="replace")

    def vocab_size(self) -> int:
        """
        Get current total size of vocabulary including base bytes, merges, and special tokens.

        Returns:
            int: Number of total entries in vocabulary.
        """
        # TODO: Return total number of vocabulary items
        return len(self.vocab)

    def get_token_bytes(self, token_id: int) -> bytes:
        """
        Retrieve underlying byte representation of a given token ID.

        Args:
            token_id (int): Token ID to look up.

        Returns:
            bytes: Byte sequence corresponding to token_id, or default placeholder if not found.
        """
        # TODO: Retrieve byte mapping from vocabulary dictionary for specified token_id
        return self.vocab.get(token_id, b"")


In [34]:
# [KEEP_IMPLEMENTATION]
def demo_byte_encoding() -> None:
    """
    Demonstrate byte-level UTF-8 encoding across various languages and character sets.

    Returns:
        None
    """
    print("=" * 60)
    print("Byte-Level Encoding")
    print("=" * 60)

    texts = [
        ("English", "hello"),
        ("Chinese", "你好"),
        ("Japanese", "こんにちは"),
        ("Emoji", "🔥🌍"),
        ("Mixed", "hello你好🔥"),
        ("Code", "def f(x):"),
    ]

    for label, text in texts:
        b = list(text.encode("utf-8"))
        print(f"{label:10s}: {len(text):2d} chars -> {len(b):2d} bytes -> {b[:16]}{'...' if len(b) > 16 else ''}")


# [KEEP_IMPLEMENTATION]
def demo_pre_tokenization() -> None:
    """
    Demonstrate GPT-2 regular expression pre-tokenization on different text formats.

    Returns:
        None
    """
    print("\n" + "=" * 60)
    print("Pre-Tokenization (GPT-2 Regex)")
    print("=" * 60)

    texts = [
        "Hello, world! Don't stop.",
        "def train(model, data):",
        "The price is $3.14 per unit.",
        "  multiple   spaces   here  ",
    ]

    for text in texts:
        chunks = pre_tokenize(text)
        print(f"\n'{text}'")
        print(f"  -> {chunks}")


# [KEEP_IMPLEMENTATION]
def demo_full_tokenizer() -> None:
    """
    Demonstrate end-to-end BPE training, special token addition, encoding, and decoding.

    Returns:
        None
    """
    print("\n" + "=" * 60)
    print("Training Production Tokenizer")
    print("=" * 60)

    corpus = (
        "The quick brown fox jumps over the lazy dog. "
        "The quick brown fox runs through the forest. "
        "Machine learning models process natural language. "
        "Machine learning transforms how we build software. "
        "Deep learning models need large datasets to train. "
        "def train(model, data): return model.fit(data) "
        "def predict(model, x): return model(x) "
        "for i in range(100): print(i) "
    )

    tok = ProductionTokenizer()
    tok.train(corpus, num_merges=50)

    bos_id = tok.add_special_token("<|begin|>")
    eos_id = tok.add_special_token("<|end|>")
    user_id = tok.add_special_token("<|user|>")
    asst_id = tok.add_special_token("<|assistant|>")

    print(f"\nVocab size: {tok.vocab_size()}")
    print(f"Special tokens: <|begin|>={bos_id}, <|end|>={eos_id}, <|user|>={user_id}, <|assistant|>={asst_id}")

    print("\n" + "=" * 60)
    print("Encoding Tests")
    print("=" * 60)

    test_texts = [
        "The quick brown fox.",
        "你好世界 Hello World",
        "🔥🌍🚀",
        "def foo(x): return x + 1",
        "<|begin|><|user|>Hello<|end|>",
        "Machine learning is powerful.",
    ]

    for text in test_texts:
        ids = tok.encode(text)
        decoded = tok.decode(ids)
        raw_bytes = len(text.encode("utf-8"))
        print(f"\nInput:   {text}")
        print(f"IDs:     {ids[:20]}{'...' if len(ids) > 20 else ''}")
        print(f"Tokens:  {len(ids)} (from {raw_bytes} bytes, ratio: {len(ids)/raw_bytes:.2f})")
        print(f"Decoded: {decoded}")
        roundtrip = "PASS" if decoded == text else "FAIL"
        print(f"Round-trip: {roundtrip}")


# [KEEP_IMPLEMENTATION]
def demo_tiktoken_comparison() -> None:
    """
    Compare custom tokenizer performance and fertility against OpenAI's tiktoken.

    Returns:
        None
    """
    try:
        import tiktoken
    except ImportError:
        print("\ntiktoken not installed. Run: pip install tiktoken")
        return

    print("\n" + "=" * 60)
    print("Comparison with tiktoken (GPT-4)")
    print("=" * 60)

    enc = tiktoken.get_encoding("cl100k_base")

    test_paragraph = "Machine learning is powerful. 机器学习很强大。 L'apprentissage automatique est puissant. 🤖💪"

    tokens = enc.encode(test_paragraph)
    pieces = [enc.decode([t]) for t in tokens]

    print(f"\nInput: {test_paragraph}")
    print(f"GPT-4 tokens ({len(tokens)}): {pieces}")

    languages = [
        ("English", "The quick brown fox jumps over the lazy dog."),
        ("Chinese", "快速的棕色狐狸跳过了懒狗。"),
        ("Japanese", "素早い茶色のキツネが怠け者の犬を飛び越えた。"),
        ("Korean", "빠른 갈색 여우가 게으른 개를 뛰어넘었다."),
        ("Code", "def quicksort(arr): return sorted(arr)"),
        ("Emoji", "🎉🎊🎈🎁🎂🎄🎃🎆🎇✨"),
    ]

    print(f"\n{'Language':<10} {'Chars':<6} {'Tokens':<7} {'Fertility':<10}")
    print("-" * 35)
    for label, text in languages:
        toks = enc.encode(text)
        words = len(text.split())
        fertility = len(toks) / max(words, 1)
        print(f"{label:<10} {len(text):<6} {len(toks):<7} {fertility:<10.2f}")


# ===== UNIT TESTS =====


def test_pre_tokenize() -> None:
    """Test pre_tokenize output structure and edge cases."""
    res = pre_tokenize("Hello world! 123")
    assert isinstance(res, list), "Pre-tokenize output must be a list of strings."
    assert len(res) > 0, "Pre-tokenize output should not be empty for non-empty text."
    assert "".join(res) == "Hello world! 123", "Concatenated pre-tokenized chunks must reconstruct original text."

    # Edge Case: empty string
    empty_res = pre_tokenize("")
    assert empty_res == [], "Edge Case Failed: Empty string must return an empty list."


def test_apply_merge() -> None:
    """Test token pair merging logic and sequence lengths."""
    seq = [10, 20, 10, 20, 30]
    merged = apply_merge(seq, (10, 20), 100)
    assert isinstance(merged, list), "apply_merge must return a list."
    assert len(merged) == 3, f"Expected merged length of 3, got {len(merged)}. Check pair replacement step."
    assert merged == [100, 100, 30], "Merged output values do not match expected replaced token IDs."

    # Edge Case: single element sequence
    single = apply_merge([10], (10, 20), 100)
    assert single == [10], "Edge Case Failed: Single element list should remain unchanged."


def test_special_token_handler() -> None:
    """Test special token pattern creation and text splitting."""
    handler = SpecialTokenHandler()
    handler.add_token("<|end|>", 500)
    parts = handler.split_with_specials("Hello<|end|>World")

    assert isinstance(parts, list), "split_with_specials must return a list."
    assert len(parts) == 3, f"Expected 3 parts after splitting, got {len(parts)}."
    assert parts[1] == ("<|end|>", True), "Special token tag flag or value is incorrect."

    # Edge Case: text with no special tokens
    no_specials = handler.split_with_specials("Plain text")
    assert no_specials == [("Plain text", False)], "Edge Case Failed: Plain text splitting mismatched."


def test_production_tokenizer_train() -> None:
    """Test tokenizer training and vocabulary expansion."""
    tok = ProductionTokenizer()
    initial_vocab_size = tok.vocab_size()
    tok.train("abc abc abc", num_merges=2)

    assert tok.vocab_size() > initial_vocab_size, "Vocabulary size should increase after training BPE merges."
    assert len(tok.merges) <= 2, "Number of recorded merges should not exceed requested num_merges."

    # Edge Case: training on empty text
    tok_empty = ProductionTokenizer()
    tok_empty.train("", num_merges=5)
    assert tok_empty.vocab_size() == 256, "Edge Case Failed: Vocabulary size should stay at 256 for empty training text."


def test_production_tokenizer_encode_decode() -> None:
    """Test tokenizer round-trip integrity, output shape, and special tokens handling."""
    tok = ProductionTokenizer()
    tok.train("The quick brown fox jumps over the lazy dog.", num_merges=10)
    tok.add_special_token("<|special|>")

    text = "The quick fox <|special|>"
    encoded = tok.encode(text)

    assert isinstance(encoded, list), "Encoder output must be a list of integers."
    assert len(encoded) > 0, "Encoded token list should not be empty."
    assert all(isinstance(idx, int) for idx in encoded), "All token IDs in encoded list must be integers."

    decoded = tok.decode(encoded)
    assert isinstance(decoded, str), "Decoder output must be a string."
    assert decoded == text, f"Round-trip decoded text '{decoded}' does not match original text '{text}'."

    # Edge Case: single character string encoding
    single_char_ids = tok.encode("A")
    assert len(single_char_ids) == 1, "Edge Case Failed: Single byte character should yield exactly 1 token ID."


if __name__ == "__main__":
    demo_byte_encoding()
    demo_pre_tokenization()
    demo_full_tokenizer()
    demo_tiktoken_comparison()

Byte-Level Encoding
English   :  5 chars ->  5 bytes -> [104, 101, 108, 108, 111]
Chinese   :  2 chars ->  6 bytes -> [228, 189, 160, 229, 165, 189]
Japanese  :  5 chars -> 15 bytes -> [227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129, 161, 227, 129, 175]
Emoji     :  2 chars ->  8 bytes -> [240, 159, 148, 165, 240, 159, 140, 141]
Mixed     :  8 chars -> 15 bytes -> [104, 101, 108, 108, 111, 228, 189, 160, 229, 165, 189, 240, 159, 148, 165]
Code      :  9 chars ->  9 bytes -> [100, 101, 102, 32, 102, 40, 120, 41, 58]

Pre-Tokenization (GPT-2 Regex)

'Hello, world! Don't stop.'
  -> ['Hello', ',', ' world', '!', ' Don', "'t", ' stop', '.']

'def train(model, data):'
  -> ['def', ' train', '(', 'model', ',', ' data', '):']

'The price is $3.14 per unit.'
  -> ['The', ' price', ' is', ' $', '3', '.', '14', ' per', ' unit', '.']

'  multiple   spaces   here  '
  -> [' ', ' multiple', '  ', ' spaces', '  ', ' here', '  ']

Training Production Tokenizer

Vocab size: 310
Special tokens: 